# JobFindr — Recommendation Engine: Complete Technical Walkthrough
**With Real Kaggle Job Data (500+ listings)**

Self-contained Google Colab notebook. No local files required.

## Architecture Overview
```
data_ingestion/job_descriptions.csv   ←  Kaggle: ravindrasinghrana/job-description-dataset
          |
          v  import_kaggle.py  OR  _seed_database() on first boot
     SQLite DB  (jobfindr.db)
          |
          v  JobRecommender.__init__()
  build_job_corpus(job) x500+
  TfidfVectorizer.fit_transform()  →  tfidf_matrix (500, vocab)
          |
  GET /recommendations?algo=hybrid
          |
          v  build_user_profile_text(user)
  query_vec  →  one of 3 algorithms  →  ranked job list
          |
          v  get_score_breakdown(user, job)
  {skills_match, role_alignment, salary_fit, seniority, demand_score}
```
## What This Notebook Covers
| # | Section |
|---|---------|
| 1 | Kaggle Dataset Setup |
| 2 | Data Exploration (real 500-job corpus) |
| 3 | Text Preprocessing Pipeline |
| 4 | TF-IDF Vectorisation |
| 5 | Algorithm 1 — TF-IDF Cosine Similarity |
| 6 | Algorithm 2 — Jaccard Keyword Overlap |
| 7 | Algorithm 3 — Hybrid |
| 8 | 5-Axis Score Breakdown |
| 9 | Resume Parser |
| 10 | End-to-End Demo |
| 11 | Evaluation (Precision@K) |
| 12 | Visualisations |
| 13 | Try Your Own Profile |

## 1. Kaggle Dataset Setup

We use the **Job Description Dataset** by Ravindra Singh Rana from Kaggle.

> **Dataset:** https://www.kaggle.com/datasets/ravindrasinghrana/job-description-dataset

### Option A — Kaggle API (Recommended)
1. Go to https://www.kaggle.com/settings → **API** → **Create New Token** → downloads `kaggle.json`
2. Run the cell below and upload your `kaggle.json` when prompted.

### Option B — Manual Upload
1. Download `job_descriptions.csv` directly from the Kaggle page above
2. Upload it to Colab using the file browser (left sidebar → upload icon)
3. Skip to **Cell 1b** and set `CSV_PATH` to your uploaded filename.

In [ ]:
# ── Cell 1a: Download via Kaggle API ─────────────────────────────────────────
!pip install -q kaggle scikit-learn pandas matplotlib tqdm

from google.colab import files
print('Upload your kaggle.json now...')
uploaded = files.upload()

import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print('Downloading dataset from Kaggle...')
!kaggle datasets download -d ravindrasinghrana/job-description-dataset --unzip -q
!ls *.csv 2>/dev/null || echo 'CSV files:' && ls
print('Done.')

In [ ]:
# ── Cell 1b: Load the CSV ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# If you uploaded manually, change this to your filename
CSV_PATH = 'job_descriptions.csv'

print(f'Loading {CSV_PATH}...')
try:
    raw_df = pd.read_csv(CSV_PATH, encoding='utf-8', on_bad_lines='skip', low_memory=False)
except (UnicodeDecodeError, TypeError):
    raw_df = pd.read_csv(CSV_PATH, encoding='latin-1', on_bad_lines='skip', low_memory=False)

print(f'Shape: {raw_df.shape}')
print(f'Columns: {list(raw_df.columns)}')
raw_df.head(2)

In [ ]:
# ── Cell 1c: Column mapping + cleaning ───────────────────────────────────────
# Auto-detect column names across different Kaggle dataset versions
def pick_col(df, aliases, default=''):
    for col in aliases:
        if col in df.columns:
            return df[col].fillna(default).astype(str)
    return pd.Series([default] * len(df), index=df.index)

TITLE_ALIASES   = ['Job Title', 'title', 'job_title', 'Position']
COMPANY_ALIASES = ['Company', 'company', 'Employer', 'Company Name']
LOC_ALIASES     = ['location', 'Location', 'Job Location']
DESC_ALIASES    = ['Job Description', 'description', 'job_description', 'Description']
SKILLS_ALIASES  = ['skills', 'Skills', 'Key Skills', 'key_skills', 'Required Skills']
TYPE_ALIASES    = ['Job Type', 'job_type', 'Employment Type']

df = pd.DataFrame({
    'title':       pick_col(raw_df, TITLE_ALIASES),
    'company':     pick_col(raw_df, COMPANY_ALIASES, 'Unknown Company'),
    'location':    pick_col(raw_df, LOC_ALIASES, 'Remote'),
    'description': pick_col(raw_df, DESC_ALIASES),
    'skills_raw':  pick_col(raw_df, SKILLS_ALIASES),
    'job_type':    pick_col(raw_df, TYPE_ALIASES, 'Full-time'),
})

# ── Department inference from title + skills ────────────────────────────────
DEPT_MAP = {
    'AI Research':      ['ai research', 'artificial intelligence research'],
    'Machine Learning': ['machine learning', 'ml engineer', 'computer vision', 'nlp', 'pytorch', 'tensorflow'],
    'Data Science':     ['data scientist', 'data science', 'analytics', 'quantitative'],
    'Data Engineering': ['data engineer', 'spark', 'kafka', 'etl', 'databricks', 'airflow'],
    'MLOps':            ['mlops', 'devops', 'ci/cd', 'kubernetes', 'docker'],
    'Backend':          ['backend', 'back-end', 'microservices', 'api', 'java', 'golang', 'ruby'],
    'Full-Stack':       ['full stack', 'fullstack', 'react', 'angular', 'vue', 'node'],
    'Frontend':         ['frontend', 'front-end', 'ios', 'android', 'swift', 'kotlin'],
    'Design':           ['designer', 'ux', 'figma', 'product design'],
    'Product':          ['product manager', 'product owner', ' pm ', 'roadmap'],
    'Security':         ['security', 'cybersecurity', 'zero trust', 'cryptography'],
    'Platform':         ['platform', 'cloud', 'aws', 'azure', 'gcp', 'terraform'],
}

def infer_dept(row):
    combined = (str(row['title']) + ' ' + str(row['skills_raw'])).lower()
    for dept, kws in DEPT_MAP.items():
        if any(kw in combined for kw in kws):
            return dept
    return 'Engineering'

df['dept'] = df.apply(infer_dept, axis=1)

# ── Filter & sample ─────────────────────────────────────────────────────────
df = df[
    (df['title'].str.strip() != '') &
    (df['title'] != 'nan') &
    (df['description'].str.len() > 80)
].reset_index(drop=True)

N_JOBS = 500
df = df.sample(n=min(N_JOBS, len(df)), random_state=42).reset_index(drop=True)
df['id'] = range(1, len(df) + 1)

# Parse skills into lists
df['skills'] = df['skills_raw'].apply(
    lambda s: [x.strip() for x in str(s).split(',') if x.strip() and x.strip() != 'nan'][:15]
)

print(f'Final dataset: {len(df)} jobs')
print(f'Dept distribution:')
print(df['dept'].value_counts().to_string())
df[['id','title','company','dept','location']].head(8)

## 2. Data Exploration

Let's understand the real Kaggle dataset before building the recommendation engine.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Kaggle Job Dataset — Overview', fontsize=14, fontweight='bold')

# Dept distribution
ax = axes[0]
dept_counts = df['dept'].value_counts()
ax.barh(dept_counts.index[:12], dept_counts.values[:12], color='#6366f1', alpha=0.85)
ax.set_title('Jobs per Department')
ax.set_xlabel('Count')
ax.grid(axis='x', alpha=0.3)

# Skills word frequency
ax = axes[1]
all_skills = [s for sublist in df['skills'].tolist() for s in sublist]
from collections import Counter
skill_counts = Counter(all_skills).most_common(15)
names, counts = zip(*skill_counts)
ax.barh(names[::-1], counts[::-1], color='#22d3ee', alpha=0.85)
ax.set_title('Top 15 Most Required Skills')
ax.set_xlabel('Job count')
ax.grid(axis='x', alpha=0.3)

# Description length histogram
ax = axes[2]
desc_lens = df['description'].str.split().str.len()
ax.hist(desc_lens, bins=40, color='#f472b6', alpha=0.85, edgecolor='white')
ax.set_title('Job Description Length')
ax.set_xlabel('Word count')
ax.set_ylabel('# Jobs')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Avg description: {desc_lens.mean():.0f} words | Median: {desc_lens.median():.0f} words')
print(f'Avg skills per job: {df["skills"].str.len().mean():.1f}')

## 3. Text Preprocessing (`app/ml/preprocessor.py`)

Three functions convert raw dicts into TF-IDF-ready strings:

| Function | Key logic |
|----------|-----------|
| `clean_text(text)` | lowercase → strip punct/URLs → tokenise → remove stop-words |
| `build_job_corpus(job)` | `title×3 + skills×3 + dept + company + description` → clean |
| `build_user_profile_text(user)` | `title×2 + skills×3 + bio` → clean |

### Why repeat title/skills 3×?
TF-IDF rewards token frequency within a document. Repeating `title` and `skills` 3× inflates
their TF counts, pushing those tokens to dominate the resulting vector — so cosine similarity
prioritises skill/role alignment over generic description noise.

### Stop-words
Custom set (no NLTK) plus job-ad noise: `job, find, opportunity, work, join, team, growth, experience`

In [ ]:
STOP_WORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with','by',
    'from','up','about','into','through','during','is','are','was','were','be',
    'been','being','have','has','had','do','does','did','will','would','could',
    'should','may','might','shall','can','need','dare','ought','used','as','this',
    'that','these','those','it','its','we','you','they','he','she','our','your',
    'their','my','his','her','which','who','whom','what','when','where','why','how',
    'all','both','each','few','more','most','other','some','such','no','not','only',
    'same','so','than','too','very','just','over','also','i','me','us',
    'job','find','opportunity','work','join','team','growth','global','experience'
}

def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = text.replace('/', ' ').replace('-', ' ')
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[!"$%&\'()*,\.:;<=>?@\[\\\]^_`{|}~]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and not t.isdigit() and len(t) > 1]
    return ' '.join(tokens)

def build_job_corpus(job):
    title  = str(job.get('title', '') or '')
    skills = ' '.join([str(s) for s in job.get('skills', [])])
    dept   = str(job.get('dept', '') or '')
    desc   = str(job.get('description', '') or '')
    comp   = str(job.get('company', '') or '')
    # title x3, skills x3 — TF-IDF frequency boost
    return clean_text(f'{title} {title} {title} {skills} {skills} {skills} {dept} {comp} {desc}')

def build_user_profile_text(user):
    title  = str(user.get('title', '') or '')
    skills = ' '.join([str(s) for s in user.get('skills', [])])
    bio    = str(user.get('bio', '') or '')
    return clean_text(f'{title} {title} {skills} {skills} {skills} {bio}')

# Demo on real Kaggle job
sample = df.iloc[0].to_dict()
print(f'=== Job Corpus (job #1: {sample["title"]}) ===')
print(build_job_corpus(sample)[:350])
print()
print(f'Tokens: {len(build_job_corpus(sample).split())}')

## 4. TF-IDF Vectorisation

$$\text{TF-IDF}(t,d) = \underbrace{\log(1+f_{t,d})}_{\text{sublinear TF}} \times \underbrace{\log\frac{N}{df_t}}_{\text{IDF}}$$

| Parameter | Value | Reason |
|-----------|-------|---------|
| `ngram_range=(1,2)` | unigrams + bigrams | captures `machine learning`, `deep learning`, `data pipeline` |
| `max_features=8000` | vocab cap | efficient for 500-job corpus |
| `sublinear_tf=True` | log(1+tf) | prevents long descriptions dominating |

With 500 real Kaggle jobs the vocabulary is far richer than 15 curated jobs — IDF can
properly penalise common words and reward rare technical terms.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

JOBS = df.to_dict('records')
corpora = [build_job_corpus(job) for job in JOBS]

# Exact config from app/ml/recommender.py
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=8000, sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform(corpora)
feature_names = vectorizer.get_feature_names_out()

print(f'TF-IDF matrix: {tfidf_matrix.shape}')
print(f'  {tfidf_matrix.shape[0]} job documents')
print(f'  {tfidf_matrix.shape[1]} unique n-gram features')
print(f'  Matrix sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.1%}')

# Top features for job #1
v0 = tfidf_matrix[0].toarray().flatten()
top_idx = np.argsort(v0)[::-1][:15]
print(f"\nTop TF-IDF terms for '{JOBS[0]['title']}' @ {JOBS[0]['company']}:")
for i in top_idx:
    print(f'  {feature_names[i]:<35} weight={v0[i]:.4f}')

## 5. Algorithm 1 — TF-IDF Cosine Similarity

$$\text{cosine}(\vec{q}, \vec{d_i}) = \frac{\vec{q} \cdot \vec{d_i}}{\|\vec{q}\| \cdot \|\vec{d_i}\|}$$

**Steps:**
1. `build_user_profile_text(user)` → query string
2. `vectorizer.transform([query])` → sparse query vector (same space as TF-IDF matrix)
3. `cosine_similarity(q, tfidf_matrix)` → scores for all 500 jobs in one operation
4. `np.argsort(scores)[::-1][:top_n]` → ranked indices
5. **Scale:** `score = min(99, max(40, round(raw × 200)))`  
   Raw cosine for good matches is ~0.15–0.5; ×200 maps to 30–99 display range.
6. **Rank-0 UX boost:** `if rank==0 and score<85: score += 20`

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_tfidf(user, jobs, vectorizer, tfidf_matrix, top_n=10):
    """Exact replica of JobRecommender._recommend_tfidf()"""
    query_text = build_user_profile_text(user)
    query_vec  = vectorizer.transform([query_text])
    sims       = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_idx    = np.argsort(sims)[::-1][:top_n]
    results = []
    for rank, idx in enumerate(top_idx):
        job = dict(jobs[idx])
        raw = float(sims[idx])
        score = int(min(99, max(40, round(raw * 200))))
        if rank == 0 and score < 85:
            score = min(99, score + 20)
        job['match_score'] = score
        job['raw_cosine']  = round(raw, 4)
        job['algo'] = 'TF-IDF'
        results.append(job)
    return results

# Demo users
USERS = [
    {'name':'Alex Chen',     'title':'AI Specialist',       'experience_years':6,
     'bio':'Experienced in machine learning, NLP, and scalable backend architecture.',
     'skills':['Python','Machine Learning','NLP','TensorFlow','PyTorch','Docker','Kubernetes','SQL']},
    {'name':'Priya Sharma',  'title':'Full Stack Developer', 'experience_years':4,
     'bio':'Full-stack developer with love for clean APIs and pixel-perfect UIs.',
     'skills':['JavaScript','TypeScript','React','Node.js','GraphQL','PostgreSQL','AWS','Docker']},
    {'name':'Marcus Johnson','title':'Data Scientist',       'experience_years':5,
     'bio':'Statistical modeling, A/B experimentation, recommendation systems at scale.',
     'skills':['Python','R','Apache Spark','SQL','Statistics','Machine Learning','Scikit-learn','Pandas']},
]

for user in USERS:
    res = recommend_tfidf(user, JOBS, vectorizer, tfidf_matrix, top_n=5)
    print(f"{user['name']} ({user['title']})")
    print(f"  Skills: {user['skills'][:5]}...")
    print(f"  {'Rank':<5} {'Title':<45} {'Company':<18} {'Raw':>8} {'Score':>6}")
    for i, j in enumerate(res):
        print(f"    #{i+1}  {j['title'][:43]:<45} {j['company'][:16]:<18} {j['raw_cosine']:>8} {j['match_score']:>6}")
    print()

## 6. Algorithm 2 — Jaccard Keyword Overlap

$$J(A,B) = \frac{|A \cap B|}{|A \cup B|}$$

- **A** = token set from user profile string
- **B** = token set from job `title` + `skills`
- **Score:** `min(99, max(35, round(jaccard × 300)))`

**Benefits on the Kaggle corpus:** With 500+ jobs, many titles and skill sets overlap in
non-obvious ways. Jaccard catches exact term matches even when TF-IDF might spread similarity
too broadly across related documents.

In [ ]:
def recommend_keyword(user, jobs, top_n=10):
    """Exact replica of JobRecommender._recommend_keyword()"""
    query_text   = build_user_profile_text(user)
    query_tokens = set(re.sub(r'[^a-z0-9 ]', ' ', query_text.lower()).split())
    scored = []
    for job in jobs:
        job_tokens  = set(' '.join([str(s) for s in job.get('skills', [])]).lower().split())
        job_tokens |= set(str(job.get('title', '')).lower().split())
        inter = query_tokens & job_tokens
        union = query_tokens | job_tokens
        jac   = len(inter) / len(union) if union else 0
        scored.append((jac, job, inter))
    scored.sort(key=lambda x: x[0], reverse=True)
    results = []
    for rank, (raw, job, overlap) in enumerate(scored[:top_n]):
        j = dict(job)
        score = int(min(99, max(35, round(raw * 300))))
        if rank == 0 and score < 80:
            score = min(99, score + 15)
        j['match_score']    = score
        j['jaccard']        = round(raw, 4)
        j['overlap_tokens'] = sorted(overlap)[:5]
        j['algo'] = 'Keyword'
        results.append(j)
    return results

for user in USERS:
    res = recommend_keyword(user, JOBS, top_n=3)
    print(f"{user['name']} ({user['title']})")
    for i, j in enumerate(res):
        print(f"  #{i+1} [{j['match_score']:>2}] {j['title'][:43]:<45} J={j['jaccard']} tokens={j['overlap_tokens'][:4]}")
    print()

## 7. Algorithm 3 — Hybrid (0.7 × TF-IDF + 0.3 × Keyword)

$$\text{hybrid} = 0.7 \times \text{tfidf\_score} + 0.3 \times \text{keyword\_score}$$

Both sub-algorithms run over **all 500 jobs** before blending — no early truncation.
With a real 500-job corpus this is still sub-second on modern hardware.

**Why 70/30?** TF-IDF captures semantic context (bigrams like `machine learning`, IDF weighting)
while Keyword rewards exact token overlap — useful for specific tool names (`pytorch`, `kafka`).

In [ ]:
def recommend_hybrid(user, jobs, vectorizer, tfidf_matrix, top_n=10):
    """Exact replica of JobRecommender._recommend_hybrid()"""
    tfidf_all   = recommend_tfidf(user, jobs, vectorizer, tfidf_matrix, top_n=len(jobs))
    keyword_all = recommend_keyword(user, jobs, top_n=len(jobs))
    t_map = {j['id']: j['match_score'] for j in tfidf_all}
    k_map = {j['id']: j['match_score'] for j in keyword_all}
    blended = []
    for job in jobs:
        t = t_map.get(job['id'], 40)
        k = k_map.get(job['id'], 35)
        blended.append((int(0.7*t + 0.3*k), job))
    blended.sort(key=lambda x: x[0], reverse=True)
    results = []
    for rank, (score, job) in enumerate(blended[:top_n]):
        j = dict(job)
        if rank == 0 and score < 85:
            score = min(99, score + 10)
        j['match_score'] = score
        j['algo'] = 'Hybrid'
        results.append(j)
    return results

import time
user = USERS[0]
t0 = time.time()
rh = recommend_hybrid(user, JOBS, vectorizer, tfidf_matrix, top_n=10)
elapsed = time.time() - t0
print(f'Top-10 hybrid results for {user["name"]} on {len(JOBS)} jobs — {elapsed*1000:.1f}ms')
print()
print(f'  {"Rank":<5} {"Score":<7} {"Title":<48} Company')
print('  ' + '-'*90)
for i, j in enumerate(rh):
    print(f'  #{i+1:<4} {j["match_score"]:>5}%  {j["title"][:46]:<48} {j["company"][:20]}')

## 8. 5-Axis Score Breakdown (`get_score_breakdown()`)

Displayed as a radar chart on `/recommendations` for each job card.

| Axis | Formula | Notes |
|------|---------|-------|
| **skills_match** | `matched / total_job_skills × 100` | Partial substring match both ways |
| **role_alignment** | cosine×220, clamped [40,99] | Per-pair TF-IDF (not batch) |
| **salary_fit** | `midpoint / 200000 × 100` | $200k = 100% benchmark |
| **seniority** | `100 − |yoe − required| × 12` | Required YOE from title keywords |
| **demand_score** | static `DEPT_DEMAND` lookup | Market heat signal |

Note: The Kaggle dataset often lacks salary data so `salary_fit` defaults to 50 when 0/0.

In [ ]:
DEPT_DEMAND = {
    'AI Research':95,'Machine Learning':92,'Data Science':88,'Data Engineering':86,
    'MLOps':85,'Platform':80,'Backend':78,'Full-Stack':75,'Frontend':74,'Product':72,'Design':70,
}

def get_score_breakdown(user, job, vectorizer, tfidf_matrix, jobs):
    """Exact replica of JobRecommender.get_score_breakdown()"""
    user_skills = {str(s).lower() for s in user.get('skills', [])}
    job_skills  = [str(s).lower() for s in job.get('skills', [])]
    yoe = user.get('experience_years', 3)

    # 1. Skills Match
    if job_skills:
        matched = sum(1 for s in job_skills if any(u in s or s in u for u in user_skills))
        skills_match = min(100, int(matched / len(job_skills) * 100))
    else:
        skills_match = 50

    # 2. Role Alignment (per-pair TF-IDF cosine)
    q_vec = vectorizer.transform([build_user_profile_text(user)])
    idx   = next((i for i, j in enumerate(jobs) if j.get('id') == job.get('id')), None)
    if idx is not None:
        sim = float(cosine_similarity(q_vec, tfidf_matrix[idx]).flatten()[0])
        role_alignment = min(99, max(40, int(sim * 220)))
    else:
        role_alignment = 60

    # 3. Salary Fit
    sal_min = job.get('salary_min', 0) or 0
    sal_max = job.get('salary_max', 0) or 0
    if sal_min == 0 and sal_max == 0:
        salary_fit = 50  # No salary data from Kaggle — neutral
    else:
        mid = (sal_min + sal_max) / 2
        salary_fit = min(100, max(20, int(mid / 200_000 * 100)))

    # 4. Seniority
    tl = str(job.get('title', '')).lower()
    if any(k in tl for k in ['principal','director']): req = 9
    elif any(k in tl for k in ['senior','staff','lead']): req = 6
    elif any(k in tl for k in ['junior','associate']): req = 1
    else: req = 3
    seniority = min(100, max(20, int(100 - abs(yoe - req) * 12)))

    # 5. Demand Score
    demand_score = DEPT_DEMAND.get(job.get('dept', ''), 70)

    return {'skills_match':skills_match, 'role_alignment':role_alignment,
            'salary_fit':salary_fit, 'seniority':seniority, 'demand_score':demand_score}

# Show breakdowns for each user's top hybrid match
for user in USERS:
    top  = recommend_hybrid(user, JOBS, vectorizer, tfidf_matrix, top_n=1)[0]
    job  = next(j for j in JOBS if j['id'] == top['id'])
    bd   = get_score_breakdown(user, job, vectorizer, tfidf_matrix, JOBS)
    print(f"{user['name']} × {job['title'][:40]} @ {job['company']}")
    for axis, val in bd.items():
        bar = chr(9608) * (val // 5)
        print(f'  {axis:<20} {val:>3}/100  {bar}')
    print()

## 9. Resume Parser (`app/ml/resume_parser.py`)

Extracts structured data from PDF resumes to auto-populate user profiles.

**Pipeline:**
1. `pdfminer.six` extracts text (fallback: `pypdf`)
2. Regex `\bskill_name\b` match against 95-item taxonomy
3. Group matched skills into 6 categories
4. Experience: `r'(\d+)\+?\s*years?\s*(?:of\s+)?(?:experience|exp)'`
5. Email extraction via standard regex

In [ ]:
SKILLS_TAXONOMY = [
    'Python','JavaScript','TypeScript','Java','C++','C#','Go','Rust','Swift','Kotlin',
    'Ruby','PHP','Scala','R','MATLAB','Bash','Shell',
    'Machine Learning','Deep Learning','NLP','Natural Language Processing',
    'Computer Vision','Reinforcement Learning','TensorFlow','PyTorch','Keras',
    'Scikit-learn','XGBoost','LightGBM','Transformers','BERT','GPT',
    'Hugging Face','LangChain','RAG','Vector Database','Embedding',
    'SQL','PostgreSQL','MySQL','MongoDB','Redis','Elasticsearch',
    'Pandas','NumPy','Spark','Hadoop','Kafka','Airflow','dbt',
    'BigQuery','Snowflake','Databricks','ETL','Data Pipeline',
    'AWS','GCP','Azure','Docker','Kubernetes','Terraform','CI/CD',
    'GitHub Actions','Jenkins','Ansible','Linux','Nginx',
    'React','Vue','Angular','Node.js','FastAPI','Flask','Django',
    'REST API','GraphQL','HTML','CSS','Tailwind',
    'Agile','Scrum','Git','JIRA','Figma','Tableau','Power BI',
    'A/B Testing','Statistics','Probability','Linear Algebra',
    'System Design','Microservices','Event-Driven Architecture',
]

SKILL_CATEGORIES = {
    'Programming Languages': ['Python','JavaScript','TypeScript','Java','C++','C#','Go','Rust','Swift','Kotlin','Ruby'],
    'AI / ML': ['Machine Learning','Deep Learning','NLP','TensorFlow','PyTorch','Keras','Scikit-learn','Transformers','BERT','GPT'],
    'Data & Databases': ['SQL','PostgreSQL','MongoDB','Redis','Pandas','NumPy','Spark','Kafka','BigQuery','Snowflake','ETL'],
    'Cloud & DevOps': ['AWS','GCP','Azure','Docker','Kubernetes','Terraform','CI/CD','GitHub Actions','Linux'],
    'Web & APIs': ['React','Vue','Angular','Node.js','FastAPI','Flask','Django','REST API','GraphQL'],
    'Soft / Other': ['Agile','Scrum','Git','Figma','Tableau','Statistics','System Design','Microservices'],
}

def parse_resume_text(text):
    tl = text.lower()
    skills = [s for s in SKILLS_TAXONOMY if re.search(r'\b' + re.escape(s.lower()) + r'\b', tl)]
    cats = {cat: [s for s in cat_skills if s in skills]
            for cat, cat_skills in SKILL_CATEGORIES.items()}
    cats = {k: v for k, v in cats.items() if v}
    exp   = re.search(r'(\d+)\s*\+?\s*years?\s*(?:of\s+)?(?:experience|exp)', tl)
    mails = re.findall(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', text)
    return {'skills_found': skills, 'skill_count': len(skills), 'categories': cats,
            'experience_years': int(exp.group(1)) if exp else None, 'emails': mails[:3]}

SAMPLE_RESUME = '''
Jane Doe  |  jane.doe@example.com
Senior Data Scientist  —  5 years of experience

Skills: Python, PyTorch, Scikit-learn, SQL, Apache Spark, AWS, Docker, Kafka, NLP

Built recommendation systems using Collaborative Filtering and A/B Testing.
Deployed REST API with FastAPI on Kubernetes with CI/CD via GitHub Actions.
Large-scale Spark pipelines on GCP feeding BigQuery and Databricks.
'''

result = parse_resume_text(SAMPLE_RESUME)
print(f"Skills found ({result['skill_count']}): {result['skills_found']}")
print(f"Experience: {result['experience_years']} years")
print(f"Emails: {result['emails']}")
print()
for cat, skills in result['categories'].items():
    print(f'  {cat}: {skills}')

## 10. End-to-End Pipeline Demo

Replicates the full `/recommendations?algo=hybrid` route flow:

```python
# app/routes/recommendations.py
user = User.query.get(session['user_id'])
algo = request.args.get('algo', 'tfidf')   # → 'tfidf' | 'keyword' | 'hybrid'
jobs = current_app.recommender.recommend_for_user(user.to_dict(), top_n=12, algo=algo)
return render_template('recommendations.html', user=user.to_dict(), jobs=jobs)
```

In [ ]:
def explain_match(query_text, job):
    """Replica of JobRecommender._explain_match() without HTML tags."""
    tokens  = set(query_text.lower().split())
    matched = [s for s in [str(x).lower() for x in job.get('skills',[])] 
               if any(t in s or s in t for t in tokens)]
    if len(matched) >= 2:
        return f"Your expertise in {', '.join(matched[:3])} aligns with this role."
    elif matched:
        return f"Your background in {matched[0]} matches {job.get('company','')}"
    top = [str(s) for s in job.get('skills',[])[:2]]
    return f"This role values {' and '.join(top)} — in high demand."

for user in USERS:
    print('='*72)
    print(f"USER: {user['name']} | {user['title']} | {user['experience_years']} YOE")
    print(f"Skills: {user['skills']}")
    print('-'*72)
    recs  = recommend_hybrid(user, JOBS, vectorizer, tfidf_matrix, top_n=5)
    query = build_user_profile_text(user)
    for i, j in enumerate(recs):
        print(f"  #{i+1} [{j['match_score']:>2}%] {j['title'][:43]:<45} @ {j['company'][:20]}")
        print(f'       -> {explain_match(query, j)}')
    top_job = next(x for x in JOBS if x['id'] == recs[0]['id'])
    bd = get_score_breakdown(user, top_job, vectorizer, tfidf_matrix, JOBS)
    print(f"  Radar: " + ' | '.join(f'{k}={v}' for k, v in bd.items()))
    print()

## 11. Evaluation — Precision@K

$$P@K = \frac{|\{\text{relevant in top-}K\}|}{K}$$

With 500 real jobs the engine is evaluated on domain-label accuracy:
a result is _relevant_ if the inferred `dept` or the `title` contains a domain keyword.

In [ ]:
def precision_at_k(recs, keywords, k=10):
    hits = sum(1 for j in recs[:k] if any(
        kw.lower() in str(j.get('title','')).lower() or
        kw.lower() in str(j.get('dept','')).lower()
        for kw in keywords
    ))
    return hits / k

tests = [
    (USERS[0], ['machine learning','ml','nlp','ai','data science'], 10),
    (USERS[1], ['full stack','frontend','react','typescript','backend','node'], 10),
    (USERS[2], ['data','statistics','recommendation','analyst','scientist'], 10),
]

print(f'{"User":<20} {"TF-IDF P@10":<16} {"Keyword P@10":<16} {"Hybrid P@10"}')
print('-'*65)
for user, kw, k in tests:
    rt = recommend_tfidf(user, JOBS, vectorizer, tfidf_matrix, top_n=k)
    rk = recommend_keyword(user, JOBS, top_n=k)
    rh = recommend_hybrid(user, JOBS, vectorizer, tfidf_matrix, top_n=k)
    print(f"{user['name']:<20} {precision_at_k(rt,kw,k):<16.0%} {precision_at_k(rk,kw,k):<16.0%} {precision_at_k(rh,kw,k):.0%}")

## 12. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'JobFindr — Match Scores on {len(JOBS)}-Job Kaggle Corpus', fontsize=14, fontweight='bold')
colors = {'TF-IDF':'#6366f1','Keyword':'#22d3ee','Hybrid':'#f472b6'}

for ax, user in zip(axes, USERS):
    rh = recommend_hybrid(user, JOBS, vectorizer, tfidf_matrix, top_n=5)
    rt = recommend_tfidf(user, JOBS, vectorizer, tfidf_matrix, top_n=len(JOBS))
    rk = recommend_keyword(user, JOBS, top_n=len(JOBS))
    labels = [j['title'][:22] for j in rh]
    hids   = [j['id'] for j in rh]
    t_map  = {j['id']: j['match_score'] for j in rt}
    k_map  = {j['id']: j['match_score'] for j in rk}
    x, w   = np.arange(len(labels)), 0.25
    ax.bar(x-w, [t_map.get(i,40) for i in hids], w, label='TF-IDF',  color=colors['TF-IDF'],  alpha=0.85)
    ax.bar(x,   [k_map.get(i,35) for i in hids], w, label='Keyword', color=colors['Keyword'], alpha=0.85)
    ax.bar(x+w, [j['match_score'] for j in rh],  w, label='Hybrid',  color=colors['Hybrid'],  alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.set_ylim(0, 105); ax.set_ylabel('Match Score')
    ax.set_title(f"{user['name']}\n({user['title']})", fontsize=10)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Radar charts — 5-axis score breakdown for each user's top match
def draw_radar(ax, scores, title):
    cats   = list(scores.keys())
    vals   = list(scores.values())
    N      = len(cats)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    vals   += vals[:1]; angles += angles[:1]
    ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats, fontsize=8)
    ax.set_ylim(0, 100)
    ax.plot(angles, vals, 'o-', lw=2, color='#6366f1')
    ax.fill(angles, vals, alpha=0.25, color='#6366f1')
    ax.set_title(title, size=9, pad=14, fontweight='bold')

fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw={'polar': True})
fig.suptitle('5-Axis Score Breakdown — Top Hybrid Match per User (Kaggle Corpus)', fontsize=12, fontweight='bold')

for ax, user in zip(axes, USERS):
    top = recommend_hybrid(user, JOBS, vectorizer, tfidf_matrix, top_n=1)[0]
    job = next(j for j in JOBS if j['id'] == top['id'])
    bd  = get_score_breakdown(user, job, vectorizer, tfidf_matrix, JOBS)
    draw_radar(ax, bd, f"{user['name']}\nvs {str(job['title'])[:22]}")

plt.tight_layout()
plt.show()

In [ ]:
# Score distribution across all 500 jobs for one user
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Score Distribution Across All {len(JOBS)} Kaggle Jobs', fontsize=13, fontweight='bold')

user = USERS[0]
rt_all = recommend_tfidf(user, JOBS, vectorizer, tfidf_matrix, top_n=len(JOBS))
rk_all = recommend_keyword(user, JOBS, top_n=len(JOBS))
rh_all = recommend_hybrid(user, JOBS, vectorizer, tfidf_matrix, top_n=len(JOBS))

ax = axes[0]
bins = range(35, 100, 5)
ax.hist([j['match_score'] for j in rt_all], bins=bins, alpha=0.7, label='TF-IDF',  color='#6366f1')
ax.hist([j['match_score'] for j in rk_all], bins=bins, alpha=0.7, label='Keyword', color='#22d3ee')
ax.hist([j['match_score'] for j in rh_all], bins=bins, alpha=0.7, label='Hybrid',  color='#f472b6')
ax.set_xlabel('Match Score'); ax.set_ylabel('# Jobs')
ax.set_title(f'Score Distribution — {user["name"]}')
ax.legend(); ax.grid(alpha=0.3)

# Raw cosine score distribution
ax = axes[1]
raw_cosines = [j['raw_cosine'] for j in rt_all]
ax.hist(raw_cosines, bins=40, color='#6366f1', alpha=0.85, edgecolor='white')
ax.set_xlabel('Raw Cosine Similarity')
ax.set_ylabel('# Jobs')
ax.set_title('Raw TF-IDF Cosine Score Distribution')
ax.axvline(np.mean(raw_cosines), color='red', linestyle='--', label=f'Mean={np.mean(raw_cosines):.3f}')
ax.axvline(np.percentile(raw_cosines, 90), color='orange', linestyle='--', label=f'P90={np.percentile(raw_cosines,90):.3f}')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 13. Try Your Own Profile

Edit `MY_PROFILE` below and run the cell.

In [ ]:
# ── Edit this dict ───────────────────────────────────────────────────────────
MY_PROFILE = {
    'name':   'Your Name',
    'title':  'Backend Engineer',
    'experience_years': 3,
    'bio':    'Building scalable microservices and cloud-native systems.',
    'skills': ['Go', 'PostgreSQL', 'Kafka', 'Docker', 'Kubernetes', 'REST API', 'AWS'],
}
# ─────────────────────────────────────────────────────────────────────────────

print(f"Recommendations for {MY_PROFILE['name']} ({MY_PROFILE['title']}, {MY_PROFILE['experience_years']} YOE)")
print(f"Skills: {MY_PROFILE['skills']}")
print(f"Corpus size: {len(JOBS)} real Kaggle jobs")
print()

for algo_name, fn in [
    ('TF-IDF',  lambda u: recommend_tfidf(u, JOBS, vectorizer, tfidf_matrix, 5)),
    ('Keyword', lambda u: recommend_keyword(u, JOBS, 5)),
    ('Hybrid',  lambda u: recommend_hybrid(u, JOBS, vectorizer, tfidf_matrix, 5)),
]:
    recs = fn(MY_PROFILE)
    print(f'--- {algo_name} ---')
    for i, j in enumerate(recs):
        print(f"  #{i+1} [{j['match_score']:>2}%] {str(j['title'])[:43]:<45} @ {str(j['company'])[:25]}")
    print()

# Radar breakdown for hybrid top pick
top = recommend_hybrid(MY_PROFILE, JOBS, vectorizer, tfidf_matrix, top_n=1)[0]
job = next(j for j in JOBS if j['id'] == top['id'])
bd  = get_score_breakdown(MY_PROFILE, job, vectorizer, tfidf_matrix, JOBS)
print(f"Score breakdown vs '{job['title']}':")
for axis, val in bd.items():
    bar = chr(9608) * (val // 5)
    print(f'  {axis:<20} {val:>3}/100  {bar}')